## Imports and Constants

In [ ]:
# ════════════════════════════════════════════════════════════
#  Imports and constants
# ════════════════════════════════════════════════════════════

# ── Standard library ──
import io
import re
import sys
import json
import time
import hashlib
import subprocess
import contextlib
from pathlib import Path
from decimal import Decimal
from collections import defaultdict, Counter
from urllib.parse import urljoin
import datetime as dt                              
from datetime import datetime, date, timezone      

# ── Third-party ──
import openactive as oa
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import ijson
from shapely.geometry import shape, Point
from shapely.strtree import STRtree

# ── Constants / helpers ──
HARVEST_DATE = date.today().isoformat()   # date the corpus was pulled

def printer(arg):                          # indented JSON, per the OA package docs
    print(json.dumps(arg, indent=4))

## SECTION 1: Convert OpenActive Provider URLS into DF
### METHOD 1: Get the table of all providers and feeds off of OpenActive's website

In [ ]:
url = "https://status.openactive.io/" # URL of all the providers and corresponding feeds from the OA website

# Get webpage
response = requests.get(url, timeout=30)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Find tables
tables = soup.find_all("table")

### Load the table into a Pandas DataFrame

In [43]:
# instantiate the data, each row of the table will be appended to this
data = []

# go through the first table
for table in tables[:1]:
    # Skip tables without headers
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    
    # Get the index for the column which "Provider" and "Feed Status" is in
    provider_index = headers.index("Provider")
    feed_index = headers.index("Feed status")
    
    # Loop through each row of the table
    for row in table.find_all("tr"):
        # For each row, get all columns
        cells = row.find_all("td")
        
        if len(cells) <= max(provider_index, feed_index):
            continue
        
        # Get the <a> marker that includes both href and anchor text
        provider_cell = cells[provider_index].find_all("a")[1]
        # Get the URL from the href
        provider_url = provider_cell.get("href")
        # Get the name through the anchor text 
        provider_name = provider_cell.text
        
        # Instantiate the "Feed status" column as feed_cell
        feed_cell = cells[feed_index]
        
        # Set the following feeds as None
        facilityUse = None
        slot = None
        sessionSeries = None
        scheduledSession = None
        event = None
        unnamedFeed = None
        
        # Loop through each <a> in feed_cell, 
        # if there is a valid link update the feeds above.
        # Otherwise, None will represent lack of feed
        for a in feed_cell.find_all("a"):
            if a.text == "FacilityUse":
                facilityUse = a.get("href")
            elif a.text == "Slot":
                slot = a.get("href")
            elif a.text == "SessionSeries":
                sessionSeries = a.get("href")
            elif a.text == "ScheduledSession":
                scheduledSession = a.get("href")
            elif a.text == "Event":
                event = a.get("href")
            elif a.text == "Unnamed Feed":
                unnamedFeed = a.get("href")
        
        # Append all the data scraped from the above code into the data list 
        data.append({
            "provider": provider_name,
            "provider_url": provider_url,
            "facilityUse": facilityUse,
            "slot": slot,
            "sessionSeries": sessionSeries,
            "scheduledSession": scheduledSession,
            "event": event,
            "unnamedFeed": unnamedFeed
        })

# Create DataFrame from the final data list after running the code above
method1_df = pd.DataFrame(data)

### Preliminary inspection of the loaded DF

In [44]:
# display the first few rows of the df
method1_df.head()

,provider,provider_url,facilityUse,slot,sessionSeries,scheduledSession,event,unnamedFeed
0,100% TO THE TOP CIC,https://topcic.bookteq.com/api/open-active/,https://topcic.bookteq.com/api/open-active/fac...,https://topcic.bookteq.com/api/open-active/slots,NaN,NaN,NaN,NaN
1,Actihire,https://actihire.bookteq.com/api/open-active/,https://actihire.bookteq.com/api/open-active/f...,https://actihire.bookteq.com/api/open-active/s...,NaN,NaN,NaN,NaN
2,Active Hartlepool,https://activehartlepool.gs-signature.cloud/Op...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,NaN,NaN
3,Active Leeds,https://activeleeds-oa.leisurecloud.net/OpenAc...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,NaN,NaN
4,Active Luton,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,https://activeluton-openactive.legendonlineser...,NaN,NaN,NaN


In [45]:
method1_df.describe()

,provider,provider_url,facilityUse,slot,sessionSeries,scheduledSession,event,unnamedFeed
count,174,174,148,147,71,38,8,9
unique,171,174,147,146,70,37,8,9
top,Chelmsford City Sports,https://topcic.bookteq.com/api/open-active/,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://opendata.leisurecloud.live/api/feeds/C...,https://bookwhen.com/api/openactive/events,http://api.letsride.co.uk/public/v1/rides
freq,2,1,2,2,2,2,1,1


### METHOD 2: Using the OpenActive Package
Store all successful and failed feed calls into method2_feeds_df

In [30]:
# Capture printed output from oa.get_feeds()
output = io.StringIO()

with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):
    feeds = oa.get_feeds()

# Store all printed warnings/errors
errors = output.getvalue()

# Extract failed URLs from error messages
failed_urls = re.findall(
    r"ERROR: Can't get dataset: (.+)",
    errors
)

# Create table rows
data = []

# Add successful feeds
for provider_url, feed_data in feeds.items():
    data.append({
        "feed_url": provider_url,
        "status": "Success"
    })

# Add failed feeds
for url in failed_urls:
    data.append({
        "feed_url": url,
        "status": "Failed",
    })

# Convert to DataFrame
method2_feeds_df = pd.DataFrame(data)

method2_feeds_df.head()

,feed_url,status
0,https://activehartlepool.gs-signature.cloud/Op...,Success
1,https://activeleeds-oa.leisurecloud.net/OpenAc...,Success
2,https://bccleisure.gs-signature.cloud/OpenActive/,Success
3,https://bewellwigan.gs-signature.cloud/OpenAct...,Success
4,https://brimhamsactive.gs-signature.cloud/Open...,Success


### Go through each row of method2_feeds_df, depending on whether it succeeded or failed in getting a url, store the feeds gained into method2_df.

In [47]:
# instantiate where each row of data will be, at the end will convert to df
data = []

# iterate through each row in the method2_feeds_df
for _, row in method2_feeds_df.iterrows():
    # handle logic for valid oa.get_feeds() outputs
    if row["status"] == "Success":
        # instantiate the following variables as None
        provider_name = None
        provider_url = None
        facilityUse = None
        slot = None
        sessionSeries = None
        scheduledSession = None
        event = None
        unnamedFeed = np.nan # changed to np.nan because for some reason the table would save "None" rather than NaN
        
        # get the current provider feeds
        current_provider_feeds = feeds[row["feed_url"]]
        # check the type of the current provider feed and update the None variables above if available
        for feed in current_provider_feeds:
            if feed["type"] == "FacilityUse":
                facilityUse = feed["url"]
            elif feed["type"] == "Slot":
                slot = feed["url"]
            elif feed["type"] == "SessionSeries":
                sessionSeries = feed["url"]
            elif feed["type"] == "ScheduledSession":
                scheduledSession = feed["url"]
            elif feed["type"] == "Event":
                event = feed["url"]
            
            # because the oa schema may or may not have "publisher_name", we use .get
            provider_name = feed.get("publisher_name", None)
            if provider_name == "": # Some publishers have provider_name set to "" which != None or NaN so we have to change it
                provider_name = None
            provider_url = feed.get("dataset_url", None) # get the provider_url 
        
        # append each of the variable as a row to the data list
        data.append({
            "provider": provider_name,
            "provider_url": provider_url,
            "facilityUse": facilityUse,
            "slot": slot,
            "sessionSeries": sessionSeries,
            "scheduledSession": scheduledSession,
            "event": event,
            "unnamedFeed": unnamedFeed
        })
    
    # handle logic for failed oa.get_feeds() outputs
    elif row["status"] == "Failed":
        # has to set all to None except provider_url, as errors only pass back a printed error message and url with no other information
        data.append({
            "provider": None,
            "provider_url": row["feed_url"],
            "facilityUse": None,
            "slot": None,
            "sessionSeries": None,
            "scheduledSession": None,
            "event": None,
            "unnamedFeed": np.nan # changed to np.nan because for some reason the table would save "None" rather than NaN
        })

# convert to DF
method2_df = pd.DataFrame(data)

# preview of DF head
method2_df.head()

,provider,provider_url,facilityUse,slot,sessionSeries,scheduledSession,event,unnamedFeed
0,Active Hartlepool,https://activehartlepool.gs-signature.cloud/Op...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,https://opendata.leisurecloud.live/api/feeds/H...,NaN,NaN
1,Active Leeds,https://activeleeds-oa.leisurecloud.net/OpenAc...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,https://opendata.leisurecloud.live/api/feeds/A...,NaN,NaN
2,Birmingham City Council,https://bccleisure.gs-signature.cloud/OpenActive/,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,NaN,NaN
3,Wigan Leisure and Culture Trust,https://bewellwigan.gs-signature.cloud/OpenAct...,https://opendata.leisurecloud.live/api/feeds/W...,https://opendata.leisurecloud.live/api/feeds/W...,https://opendata.leisurecloud.live/api/feeds/W...,https://opendata.leisurecloud.live/api/feeds/W...,NaN,NaN
4,Brimhams Active,https://brimhamsactive.gs-signature.cloud/Open...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,https://opendata.leisurecloud.live/api/feeds/B...,NaN,NaN


### Section 1 Outputs Evaluation

In [ ]:
# Just a bunch of print methods, pretty self explanatory
print(
    "Method 1: Evaluation\n" 
    
    f"Number of Providers: {len(method1_df)}\n"
    
    f"Number of Missing Provider Names: {len(method1_df[method1_df["provider"].isna()])}\n" 
        
    f"Number of Providers with no URLs: {len(method1_df[
    method1_df[
        ["facilityUse", "slot", "sessionSeries", "scheduledSession", "event", "unnamedFeed"]
    ].isna().all(axis=1)])}\n\n" 

    "Method 2: Evaluation\n" 
    f"Number of Providers: {len(method2_df)}\n"
        
    f"Number of Missing Provider Names: {len(method2_df[method2_df["provider"].isna()])}\n"
    
    f"Number of Providers with no URLS: {len(method2_df[
    method2_df[
        ["facilityUse", "slot", "sessionSeries", "scheduledSession", "event", "unnamedFeed"]
    ].isna().all(axis=1)])}"
    )


# Get the providers that aren't in each others tables

# Can refactor this code to not duplicate

Method 1: Evaluation
Number of Providers: 174
Number of Missing Provider Names: 0
Number of Providers with no URLs: 9

Method 2: Evaluation
Number of Providers: 173
Number of Missing Provider Names: 23
Number of Providers with no URLS: 14


## SECTION 2: Harvest each URL
We first reshape the DF wide df to a long df. According to google, here's a quick definition:
- Wide data stores information about a single entity across a single row. (Good for human readability)
- Long data stores each attribute of an entity in a single dedicated row, resulting in more rows but less columns (Good for databases and processing)

In [ ]:
# Select the columns we want from our df
feed_cols = ["facilityUse", "slot", "sessionSeries", "scheduledSession", "event", "unnamedFeed"]

# create feeds_long feed by converting our original wide df
feeds_long = (
    # unpack my feed columns
    method1_df.melt(id_vars=["provider", "provider_url"],   # .melt() takes a wide df and turns it into long format
            value_vars=feed_cols,   # select the variables we want to unpack (our feed_cols)
            var_name="feed_type", value_name="feed_url")    # feed_type is unpacked into rows, with feed_url being their values 
    .dropna(subset=["feed_url"])    # NaN = that provider has no feed of that type so we drop 
    .reset_index(drop=True)     # reset the index as the melt may have altered it
)

# function to convert the text into a URL-friendly identifier
def slug(s):
    return re.sub(r"[^a-z0-9]+", "-", str(s).lower()).strip("-")

# create the feed_id
feeds_long["feed_id"] = feeds_long["provider"].map(slug) + "--" + feeds_long["feed_type"].map(slug)

# print to display the full feeds_long df
print(feeds_long.to_string())

                                                             provider                                                                       provider_url         feed_type                                                                                            feed_url                                                                         feed_id
0                                                 100% TO THE TOP CIC                                        https://topcic.bookteq.com/api/open-active/       facilityUse                                            https://topcic.bookteq.com/api/open-active/facility-uses                                                 100-to-the-top-cic--facilityuse
1                                                            Actihire                                      https://actihire.bookteq.com/api/open-active/       facilityUse                                          https://actihire.bookteq.com/api/open-active/facility-uses                            

### Function to create and return a configured HTTP session that can be reused for multiple web requests
A session is like a persistent browser connection, allowing for repeated requests to be faster, it also stores settings (e.g., headers) across requests, and avoids creating new connections everytime you call requests.get().

In [ ]:
def make_session():
    s = requests.Session() 
    s.headers.update({'User-Agent': 'ls-openactive-ws1/0.1 (UoB MSc Data Science group project)'}) # tells the server who's making the request
    # create a retry policy with a max retry of 5 times
    # backoff_factor controls the wait time between retries (it increases after each failure, roughly 1s, 2s, 4s, 8s, 16s)
    # This backoff factor is used as repeatedly calling a failing server may have adverse results
    # status_forcelist specifies which HTTP status codes should trigger retries
    retry = Retry(total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504])
    s.mount('https://', HTTPAdapter(max_retries=retry))  # attaches the retry policy to each HTTPS requests made through the session
    return s

### Function for acquiring feed from each HTTP sessions

In [ ]:
# function for acquiring the data from each of the feed_url's we got in section 1
def acquire_feed(start_url, feed_id, snapshot_id, session, out_dir,
                max_pages=2000, sleep=0.25):
    """Phase 1: page to the live edge, saving raw bytes + one pages-row per page.
    Does NOT build current state — replay reads the saved bytes for that."""
    
    # Creates a folder and set where the outputs should go
    raw_dir = Path(out_dir) / snapshot_id / "raw" / feed_id
    raw_dir.mkdir(parents=True, exist_ok=True)

    # set up initial states for some variables
    # each row of page_rows will hold every page that was downloaded
    # resume_cursor is used when the code needs to resume later 
    pages_rows, resume_cursor = [], None
    url, last_url, page_idx = start_url, None, 0    # our initial starting url, last url, and page index

    # loops while there is a URL, the URL hasn't repeated, and we haven't reached the maximum number of pages.
    while url and url != last_url and page_idx < max_pages:
        retrieved_at = dt.datetime.now(timezone.utc).isoformat()    # keeps track of the exact time of retrieval
        try:
            resp = session.get(url, timeout=60)     # sends a HTTP request to the current URL, and fails after 60s to prevent waiting forever
        
        # this code runs if something goes wrong when doing the HTTP request
        except requests.RequestException as e:
            # the following is appended to pages_row which talks about who failed and how
            pages_rows.append({"feed_id": feed_id, "snapshot_id": snapshot_id,
                "page_index": page_idx, "requested_url": url, "final_url": None,
                "http_status": None, "retrieved_at": retrieved_at, "sha256": None,
                "byte_path": None, "item_count": None, "next_url": None,
                "failure": repr(e)})
            break

        # the following only runs on successful requests
        raw = resp.content  # get the raw response from the content 
        sha = hashlib.sha256(raw).hexdigest()   # this creates a unique id for the downloaded content, to verify if downloaded content is the same
        
        byte_path = raw_dir / f"page_{page_idx:05d}.bytes"  # creates a file for where the raw response is saved in a 5 digit format (:05d)
        byte_path.write_bytes(raw)  # writes the raw response to save the original data

        payload = json.loads(raw)   # takes the raw JSON data and converts to a python dict
        items, nxt = payload.get("items", []), payload.get("next")  # extract the items on the current page and the URL for the next page
        nxt = urljoin(url, nxt) if nxt else None    # turn the next page URL into a full URL if next page is available

        # append the recorded information from the downloaded page to page_rows
        pages_rows.append({"feed_id": feed_id, "snapshot_id": snapshot_id,
            "page_index": page_idx, "requested_url": url, "final_url": resp.url,
            "http_status": resp.status_code, "retrieved_at": retrieved_at,
            "sha256": sha, "byte_path": str(byte_path), "item_count": len(items),
            "next_url": nxt, "failure": None})

        # SPECIAL CASE: if the page has no items, but the API tells us to request the exact same URL again, then stop
        if not items and nxt == url:
            resume_cursor = nxt
            break
        
        last_url, url = url, nxt    # set last_url to the current url, and url to the next url
        page_idx += 1   # increase the page index by 1
        
        if sleep:   # if we set the sleep variable from our function, then wait for that set amount of time before continuing
            time.sleep(sleep)  

    return pages_rows, resume_cursor

### Check for the above function's output using the first row of long_feeds

In [ ]:
session = make_session()

row = feeds_long.iloc[0]
pages_rows, resume_cursor = acquire_feed(
    start_url=row.feed_url,
    feed_id=row.feed_id,
    snapshot_id="H1",          
    session=session,
    out_dir="snapshots",
)

pages_df = pd.DataFrame(pages_rows)
print(len(pages_df), "pages | resume:", resume_cursor)
pages_df.head()

2 pages | resume: https://topcic.bookteq.com/api/open-active/facility-uses?afterTimestamp=1769005762689790&afterId=c696fe58-bb01-4988-bf90-b1b518ba5952


,feed_id,snapshot_id,page_index,requested_url,final_url,http_status,retrieved_at,sha256,byte_path,item_count,next_url,failure
0,100-to-the-top-cic--facilityuse,H1,0,https://topcic.bookteq.com/api/open-active/fac...,https://topcic.bookteq.com/api/open-active/fac...,200,2026-07-20T11:43:04.978130+00:00,b18b858242d5d4ba61a15674838310c7330e379d1bd1a8...,snapshots\H1\raw\100-to-the-top-cic--facilityu...,2,https://topcic.bookteq.com/api/open-active/fac...,None
1,100-to-the-top-cic--facilityuse,H1,1,https://topcic.bookteq.com/api/open-active/fac...,https://topcic.bookteq.com/api/open-active/fac...,200,2026-07-20T11:43:05.923072+00:00,4d129fb176bd56adbb85a8e71f53f25f291e69780e8d2b...,snapshots\H1\raw\100-to-the-top-cic--facilityu...,0,https://topcic.bookteq.com/api/open-active/fac...,None


### Check for the output for the corresponding bytes

In [ ]:
print(Path(pages_df.iloc[0]["byte_path"]).read_text(encoding="utf-8")[:1500])

{"next":"https:\/\/topcic.bookteq.com\/api\/open-active\/facility-uses?afterTimestamp=1769005762689790&afterId=c696fe58-bb01-4988-bf90-b1b518ba5952","items":[{"state":"updated","kind":"IndividualFacilityUse","id":"5a6a8151-ff5e-44db-8817-579c4500378b","modified":1737053360866773,"data":{"@type":"IndividualFacilityUse","@context":["https:\/\/openactive.io\/","https:\/\/openactive.io\/ns-beta"],"@id":"https:\/\/topcic.bookteq.com\/api\/open-active\/a83c656b-2524-4edb-a55e-aa2b7a52a9ca\/facility-uses\/5a6a8151-ff5e-44db-8817-579c4500378b","identifier":"5a6a8151-ff5e-44db-8817-579c4500378b","name":"2hr slot - 5\/6 a-side Outdoor Football Pitch ","url":"https:\/\/widget.bookteq.com\/topcic\/book-online\/56aebe10-4765-4c94-8a53-2d295ea881ab","facilityType":[{"@type":"Concept","@id":"https:\/\/openactive.io\/facility-types#da364f9b-8bb2-490e-9e2f-1068790b9e35","inScheme":"https:\/\/openactive.io\/facility-type","prefLabel":"Sports Hall"}],"hoursAvailable":[{"@type":"OpeningHoursSpecification"

## Run each row through the functions above to produce our pages table

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  ⚠️  SLOW CELL — full H1 harvest, ~6.5 HOURS. DO NOT RERUN CASUALLY.  ⚠️
# ----------------------------------------------------------------------
#  This re-downloads every feed from scratch and overwrites pages.csv +
#  all raw bytes under snapshots/H1/raw/.
#
#  To CONTINUE working after a kernel restart, DON'T run this — instead
#  load the saved output from disk:
# ══════════════════════════════════════════════════════════════════════

# all_pages will eventually hold information from every page downloaded from the feed
all_pages, cursors = [], {}
feeds_to_harvest = feeds_long           # swap in .head(20) if you want a smaller trial first
                                        # alternatively you can test on a specific feed type first

# loop through every feed in feeds_to_harvest
for i, row in enumerate(feeds_to_harvest.itertuples(index=False), 1):       # .itertuples() is used to allow us to access each df row with dot notation
    
    # tries to download the feed using our previously defined acquire_feed() function. Return the results to the all_pages list
    try:
        rows, cursor = acquire_feed(
            start_url=row.feed_url,
            feed_id=row.feed_id,
            snapshot_id="H1",
            session=session,
            out_dir="snapshots",
        )
        all_pages.extend(rows)
        cursors[row.feed_id] = cursor
        
    # Handles unsuccessful feeds that fail
    except Exception as e:
        # a single bad feed must never kill the whole harvest, so we record it and move on
        all_pages.append({"feed_id": row.feed_id, "snapshot_id": "H1",
            "page_index": -1, "requested_url": row.feed_url, "final_url": None,
            "http_status": None,
            "retrieved_at": dt.datetime.now(dt.timezone.utc).isoformat(),
            "sha256": None, "byte_path": None, "item_count": None,
            "next_url": None, "failure": f"harvest-crash: {e!r}"})
    
    # a tracker that updates and prints a message after every 25 feeds have been downloaded
    if i % 25 == 0:
        print(f"  {i}/{len(feeds_to_harvest)} feeds done...")

pages_all = pd.DataFrame(all_pages)     # converts the final results of all_pages into a DF
Path("snapshots/H1").mkdir(parents=True, exist_ok=True)     # creates a folder to make sure the output folder exists
pages_all.to_csv("snapshots/H1/pages.csv", index=False)     # save our outputs as csv in the folder

# drop each feed's live-edge resume cursor back onto the ledger
feeds_long["resume_cursor"] = feeds_long["feed_id"].map(cursors)

# final print to show preliminary stats of the process
print(len(pages_all), "page-rows |",
    pages_all["feed_id"].nunique(), "feeds |",
    pages_all["failure"].notna().sum(), "failure rows")

  25/421 feeds done...
  50/421 feeds done...
  75/421 feeds done...
  100/421 feeds done...
  125/421 feeds done...
  150/421 feeds done...
  175/421 feeds done...
  200/421 feeds done...
  225/421 feeds done...
  250/421 feeds done...
  275/421 feeds done...
  300/421 feeds done...
  325/421 feeds done...
  350/421 feeds done...
  375/421 feeds done...
  400/421 feeds done...
8064 page-rows | 413 feeds | 82 failure rows


In [ ]:
# 1. What kinds of failures? (the shape of your coverage gaps)
print(pages_all[pages_all["failure"].notna()]["failure"].value_counts().head(10))

# 2. Biggest feeds — confirm nothing hit the 2000 cap (cycling), just genuinely large
print(pages_all.groupby("feed_id")["page_index"].max().sort_values(ascending=False).head(10))

# 3. The 8 feeds that produced no rows at all (crashed before logging)
missing = set(feeds_long["feed_id"]) - set(pages_all["feed_id"])
print(len(missing), "feeds with zero rows:", list(missing)[:10])

failure
harvest-crash: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')                                                                                                                                                                                                                                                                                                       73
RetryError(MaxRetryError("HTTPSConnectionPool(host='gll-openactive.legendonlineservices.co.uk', port=443): Max retries exceeded with url: /api/facility-uses (Caused by ResponseError('too many 500 error responses'))"))                                                                                                                                                          1
RetryError(MaxRetryError("HTTPSConnectionPool(host='horizon-openactive.legendonlineservices.co.uk', port=443): Max retries exceeded with url: /api/facility-uses/events (Caused by ResponseError('too many 500 error responses'))"))                  

In [ ]:
# save our updated feeds_long df to csv as feeds_ledger.csv
feeds_long.to_csv("snapshots/H1/feeds_ledger.csv", index=False)

## SECTION 3: Read the harvest files and reconstruct
This code does not make any more requests, but instead uses our harvested .bytes files

In [ ]:
def replay_feed(feed_id, pages_df):
    """Fold one feed's saved pages into current state. Reads only saved bytes, no network."""
    feed_pages = (pages_df[(pages_df["feed_id"] == feed_id) &   # finds pages that belong to the feed_id passed in
                            (pages_df["byte_path"].notna())]    # skip failed pages
                .sort_values("page_index"))
    
    if len(feed_pages) == 0:    # handles feeds with no usable pages
        return [], [], {"feed_id": feed_id, "pages_read": 0, "items_seen": 0,
                        "records": 0, "tombstones": 0, "skipped_no_id": 0}

    snapshot_id = feed_pages["snapshot_id"].iloc[0]
    state, items_seen, skipped = {}, 0, 0

    # loop through every saved page
    for byte_path in feed_pages["byte_path"]:
        payload = json.loads(Path(byte_path).read_bytes())  # read the raw file
        # loop through each item in the file
        for item in payload.get("items", []):
            items_seen += 1     # keeps track of how many items we've seen
            item_id = item.get("id")    
            # skip items with no IDs
            if item_id is None:
                skipped += 1    # keep track of how many we skipped
                continue        # continue, as having no item_id doesn't necessarily mean it's the last item
            state[item_id] = item   # store items in state. If it encounters the same ID multiple times, it assumes the last version encountered is the newest.
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
#  ⚠️  For Clarence's verification. I'm not sure if the last version or the first version is the newest. May need to check that.  ⚠️
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

    # create empty list for records_rows (items that are still live/active) and tombstone_rows (items marked as deleted)
    records_rows, tombstones_rows = [], []
    # loop through each item in state
    for item_id, item in state.items():
        # base saves all the common information between live and deleted items
        base = {"feed_id": feed_id, "snapshot_id": snapshot_id, "item_id": item_id,
                "kind": item.get("kind"), "modified": str(item.get("modified"))}
        
        # if item is deleted append to tombstone
        if item.get("state") == "deleted":
            tombstones_rows.append(base)
        
        # otherwise append to records_rows
        else:
            base["data"] = item.get("data")   
            records_rows.append(base)

    # creates summary statistics of what happened for this feed
    stats = {"feed_id": feed_id, "pages_read": len(feed_pages), "items_seen": items_seen,
            "records": len(records_rows), "tombstones": len(tombstones_rows),
            "skipped_no_id": skipped}
    return records_rows, tombstones_rows, stats


# run across every feed
all_records, all_tombstones, replay_stats = [], [], []
for feed_id in pages_all["feed_id"].unique():
    recs, tombs, st = replay_feed(feed_id, pages_all)
    all_records.extend(recs)
    all_tombstones.extend(tombs)
    replay_stats.append(st)

# save all records and tombstones
out = Path("snapshots/H1")
with (out / "records.jsonl").open("w", encoding="utf-8") as f:
    for r in all_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with (out / "tombstones.jsonl").open("w", encoding="utf-8") as f:
    for t in all_tombstones:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

# print the summary statistics and preliminary preview
stats_df = pd.DataFrame(replay_stats)
print(len(all_records), "live records |", len(all_tombstones), "tombstones |",
    stats_df["feed_id"].nunique(), "feeds")
stats_df.sort_values("records", ascending=False).head()

2329014 live records | 743537 tombstones | 413 feeds


,feed_id,pages_read,items_seen,records,tombstones,skipped_no_id
362,better--scheduledsession,853,425949,390542,35407,0
162,better--slot,727,362855,200912,161943,0
408,gll--unnamedfeed,365,181613,159328,20759,0
239,places-leisure--slot,346,172111,154016,18065,0
371,everyone-active--scheduledsession,342,169748,142354,17828,0


### SANITY CHECK: Quick debugging and inspection. 
This is done because of the high tombstones to records ratio in better--slot as shown in the table above. The following code essentially asks:

"For the better--slot feed, did any item IDs appear multiple times with different states? If so, show me a few examples and tell me what the final state was."

⚠️⚠️⚠️ (Need to write up about the findings of this) ⚠️⚠️⚠️

In [ ]:
feed = "better--slot"
seen = defaultdict(list)
fp = pages_all[(pages_all.feed_id==feed) & pages_all.byte_path.notna()].sort_values("page_index")
for bp in fp.byte_path:
    for it in json.loads(Path(bp).read_bytes()).get("items", []):
        if it.get("id"): seen[it["id"]].append(it.get("state"))

flipped = {k:v for k,v in seen.items() if len(set(v)) > 1}
print(f"{len(flipped)} ids appear with >1 state; example histories:")
for k in list(flipped)[:5]:
    print(" ", seen[k][-3:], "→ final:", seen[k][-1])   # last state = what replay uses

0 ids appear with >1 state; example histories:


### Data Quality Audit
checking for each type of record, how many records have location coordinates, how many have an address, and how many have neither?

⚠️⚠️⚠️ (Need to write up about the findings of this) ⚠️⚠️⚠️

In [ ]:
def loc_block(data):    # function for finding the location
    loc = data.get("location") if isinstance(data, dict) else None
    if isinstance(loc, list):
        loc = loc[0] if loc else None
    return loc if isinstance(loc, dict) else None

def has_coords(data):   # function for finding the coordinates
    loc = loc_block(data)
    geo = loc.get("geo") if loc else None
    return isinstance(geo, dict) and geo.get("latitude") is not None and geo.get("longitude") is not None

def has_address(data):  # function for finding the address
    loc = loc_block(data)
    return bool(loc and loc.get("address") is not None)

counts = {}
with open("snapshots/H1/records.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        kind = rec.get("kind")
        data = rec.get("data", {})
        c = counts.setdefault(kind, Counter())
        c["total"] += 1
        if has_coords(data):  c["has_coords"] += 1
        if has_address(data): c["has_address"] += 1
        if not has_coords(data) and not has_address(data): c["neither"] += 1

audit = pd.DataFrame(counts).T.fillna(0).astype(int)
audit["pct_coords"] = (100 * audit["has_coords"] / audit["total"]).round(1)
print(audit)

                                 total  has_coords  has_address  neither  \
IndividualFacilityUse             3760        3760         3760        0   
FacilityUse                       6698        6026         6698        0   
IndividualFacilityUse/Slot      688242           0            0   688242   
FacilityUse/Slot                383260           0            0   383260   
SessionSeries                   122116      106107       121741      375   
ScheduledSession.SessionSeries   79145           0            0    79145   
ScheduledSession                746621          69           69   746552   
Event                           169582      169283       169439      143   
event                            71500       71500        71500        0   
session                          57746           0            0    57746   
CourseInstance                     101         101          101        0   
league                             243           0            0      243   

           

### Convert the large geojson of UK local authority district (LAD) boundaries and filtering it to London only LAD

In [ ]:
SRC = "boundaries/Local_Authority_Districts_DEC_2025_Boundaries_UK_BFC_5780731924739583250.geojson" # locate our downloaded London boundaries file
Path("boundaries").mkdir(exist_ok=True)

def _dec(o):    # define a helper for decimal values since json can't write ijson's decimals
    if isinstance(o, Decimal):
        return float(o)
    raise TypeError(f"not serializable: {type(o)}")

# create an empty list of london which will eventually hold geographic features that belong to london areas
london = []
with open(SRC, "rb") as f:
    for feat in ijson.items(f, "features.item"):
        if str(feat["properties"]["LAD25CD"]).startswith("E09"):    # LAD25CD is the 2025 local authority district codes, E09 corresponds to areas in London
            london.append(feat)

# print for the outcomes of this code
print(len(london), "London authorities (expect 33)")
with open("boundaries/london_boroughs.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": london}, f, default=_dec)   # + default=_dec

33 London authorities (expect 33)


### SANITY CHECK: Quick inspection of the geojson
Checks how many borough features there are and what are the names of the first 5 boroughs alphabetically

In [ ]:
gj = json.load(open("boundaries/london_boroughs.geojson"))  # load the geojson
print(len(gj["features"]), "boroughs")
print(sorted(f["properties"]["LAD25NM"] for f in gj["features"])[:5])

33 boroughs
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley']


## Take the reconstructed records and assign corresponding London borough depending on their coordinate data

In [ ]:
# load the 33 boroughs, convert GeoJSON into Shapely shapes, and borough names
gj = json.load(open("boundaries/london_boroughs.geojson"))
polys = [shape(ft["geometry"]) for ft in gj["features"]]
names = [ft["properties"]["LAD25NM"] for ft in gj["features"]]
tree = STRtree(polys)   # STRtree() creates a spacial index which makes geographic searches much faster

# function for returning a london borough depending on the coordinates passed in
def borough_of(lat, lng):
    pt = Point(lng, lat)    
    # loop through each borough until the coordinates match, then returns the name
    for i in tree.query(pt):                   
        if polys[i].covers(pt):
            return names[i]
    # if no name was found, then return None
    return None

# function for defining a valid coord that is located roughly in UK
def valid(lat, lng):
    if lat == 0 and lng == 0: return False  # quick check for if it's no location 
    return 49.8 <= lat <= 60.9 and -8.65 <= lng <= 1.8  # the rough UK-wide check

# function for extracting local information from a record. Returns latitude, longitude, and whether address exists.
def extract_point(data):
    # if location is a dictionary, get it's location field otherwise return None
    loc = data.get("location") if isinstance(data, dict) else None  
    # if location is a list instead, get it's location field otherwise return None
    if isinstance(loc, list):   
        loc = loc[0] if loc else None 
    # checks if location is a dictionary. If there isn't a usable location object, then stop
    if not isinstance(loc, dict): 
        return None
    geo = loc.get("geo")    # get the geographic information
    # check again if the geographic information is a dictionary to protect against unexpected data
    if not isinstance(geo, dict): 
        return None
    # try extract a tuple containing the latitude, longitude, and address
    try:    
        return float(geo["latitude"]), float(geo["longitude"]), loc.get("address") is not None
    # reutrn None if an error occurs
    except (TypeError, ValueError, KeyError): 
        return None

# create outputs container, rows will contain the results for records with usable coordinates and n_no_coord is a counter for coords we couldn't extract
rows, n_no_coord = [], 0
with open("snapshots/H1/records.jsonl", encoding="utf-8") as f:
    # read every line of our records.jsonl saved earlier
    for line in f:
        rec = json.loads(line)  # convert each JSONL line into a python dictionary
        p = extract_point(rec.get("data", {}))  # pass the record data to the extraction_point function

        if p is None:   # if no coordinate add to the n_no_coord count
            n_no_coord += 1
            continue
        
        # for valid p, we extract the lat, lng, and has_addr
        lat, lng, has_addr = p
        
        if not valid(lat, lng): # we check if the latitude or longitude are valid with our valid() function
            cls, boro = "scope-indeterminate", None # if invalid, classification is "scope-indeterminate" and the borough is None
            
        # if the valid() function returns valid information we run the following
        else:
            boro = borough_of(lat, lng) # we try get the london borough of these coordinates
            cls = "known-inside" if boro else "known-outside"   # 'known-inside' if the coords are in london and 'known-outside' otherwise
        
        # append the data to rows (reminder: holds results of records with usable coords)
        rows.append({"item_id": rec["item_id"], "feed_id": rec["feed_id"], "kind": rec.get("kind"),
                    "lat": lat, "lng": lng, "has_address": has_addr,
                    "borough": boro, "classification": cls})

geo = pd.DataFrame(rows)    # create the final df
geo.to_csv("snapshots/H1/geo.csv", index=False)     # save this final df

# print the preview/results of this processes
print(f"{len(geo):,} located | {n_no_coord:,} await inheritance")   # pritns the number of located records
print(geo["classification"].value_counts())     # print that tells us how the records are classified
print(geo[geo.classification == "known-inside"]["borough"].value_counts())      # prints the counts of records per borough

356,846 located | 1,972,168 await inheritance
classification
known-outside          260661
known-inside            95511
scope-indeterminate       674
Name: count, dtype: int64
borough
Waltham Forest            7904
Greenwich                 6505
Hillingdon                6468
Camden                    5281
Barnet                    5255
Newham                    5085
Enfield                   4921
Islington                 4637
Hackney                   4542
Croydon                   4266
Lewisham                  3377
Westminster               2925
Southwark                 2897
Ealing                    2855
Hammersmith and Fulham    2669
Richmond upon Thames      2666
Havering                  2627
Merton                    2492
Brent                     2176
Sutton                    2053
Barking and Dagenham      1789
Kensington and Chelsea    1550
Wandsworth                1504
Bromley                   1463
Tower Hamlets             1445
Bexley                    1410
Lambeth  

### SANITY CHECK: Checking the contribution of records from Haringey and Redbridge
Quick check because Haringey and Redbridge have very few records despite having borough specific providers (e.g., Haringey Council & Redbridge Sports Centre)

⚠️⚠️⚠️ (Need to write up about the findings of this) ⚠️⚠️⚠️

In [124]:
geo[geo.borough.isin(["Haringey", "Redbridge"])]["feed_id"].value_counts().head(10)

feed_id
british-cycling--unnamedfeed            339
haringey-council--facilityuse            48
redbridge-sports-centre--facilityuse     41
open-sessions--sessionseries             26
played--sessionseries                    24
bookwhen--sessionseries                   8
good-gym--event                           8
teamup--sessionseries                     6
vision-redbridge--sessionseries           2
Name: count, dtype: int64

# SECTION 4: Summary and Audit of H1 Snapshot

In [ ]:
SNAP = "H1" # identifies the snapshot (H1, H2, or H3, H2-3 have not been implemented yet)
out = Path("snapshots") / SNAP

# acquisition (from ledger + pages) 
feeds = pd.read_csv(out / "feeds_ledger.csv")
pages = pd.read_csv(out / "pages.csv")
n_attempted  = feeds["feed_id"].nunique()
n_failed     = pages[pages["failure"].notna()]["feed_id"].nunique()
n_converged  = int(feeds["resume_cursor"].notna().sum()) if "resume_cursor" in feeds else None
n_pages      = int(pages["byte_path"].notna().sum())
ts = pd.to_datetime(pages["retrieved_at"], errors="coerce")

# reconstruction S0 (count lines, no full load)
def count_lines(p):
    with open(p, "rb") as f: return sum(1 for _ in f)
n_records    = count_lines(out / "records.jsonl")
n_tombstones = count_lines(out / "tombstones.jsonl")

# geography 
geo = pd.read_csv(out / "geo.csv")
gc  = geo["classification"].value_counts().to_dict()
located   = len(geo)
no_coord  = n_records - located # mostly inheritance-pending children

# provenance
def git_sha():
    try: return subprocess.check_output(["git","rev-parse","--short","HEAD"]).decode().strip()
    except Exception: return None

manifest = {
    "snapshot_id": SNAP,
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_sha": git_sha(),
    "python": sys.version.split()[0],
    # acquisition
    "feeds_attempted":  int(n_attempted),
    "feeds_converged":  n_converged,
    "feeds_failed":     int(n_failed),
    "pages_acquired":   n_pages,
    "collection_start": ts.min().isoformat() if pd.notna(ts.min()) else None,
    "collection_end":   ts.max().isoformat() if pd.notna(ts.max()) else None,
    # reconstruction (S0)
    "records_s0":  n_records,
    "tombstones":  n_tombstones,
    # geography (of located records)
    "located":               located,
    "known_inside_london":   int(gc.get("known-inside", 0)),
    "known_outside_london":  int(gc.get("known-outside", 0)),
    "scope_indeterminate":   int(gc.get("scope-indeterminate", 0)),
    "no_coord_s0":           int(no_coord),
    # downstream — not this stream's to compute at S0
    "lineage_metrics":  "pending S1 (Michael)",
    "schedule_metrics": "pending S1b (Michael)",
}

# reconciliation checks — the numbers must add up
assert manifest["known_inside_london"] + manifest["known_outside_london"] + \
        manifest["scope_indeterminate"] == located, "geo classes != located"
assert located + no_coord == n_records, "located + no_coord != records"

with open(out / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

C:\Users\wngnc\AppData\Local\Temp\ipykernel_30332\2177230547.py:25: DtypeWarning: Columns (0: item_id) have mixed types. Specify dtype option on import or set low_memory=False.
  geo = pd.read_csv(out / "geo.csv")


{
  "snapshot_id": "H1",
  "built_at_utc": "2026-07-20T19:46:41.115783+00:00",
  "git_sha": "78ffa09",
  "python": "3.13.5",
  "feeds_attempted": 413,
  "feeds_converged": 334,
  "feeds_failed": 82,
  "pages_acquired": 7982,
  "collection_start": "2026-07-20T11:53:48.513410+00:00",
  "collection_end": "2026-07-20T18:26:24.127890+00:00",
  "records_s0": 2329014,
  "tombstones": 743537,
  "located": 356846,
  "known_inside_london": 95511,
  "known_outside_london": 260661,
  "scope_indeterminate": 674,
  "no_coord_s0": 1972168,
  "lineage_metrics": "pending S1 (Michael)",
  "schedule_metrics": "pending S1b (Michael)"
}


⚠️⚠️⚠️ (NEED TO IMPROVE FOR H2 and H3 USAGE) ⚠️⚠️⚠️